In [1]:
import MDAnalysis as mda
import pandas as pd
from collections import defaultdict
from MDAnalysis.lib.distances import distance_array

# Load system
u = mda.Universe("6wto_no_water.prmtop", "6wto_pca_minima1.dcd")
protein = u.select_atoms("protein")
cutoff = 3.5
total_frames = len(u.trajectory)

# Ligand selections
selections = {
    "Whole": "resname 3JW and not name H*",
    "Selected": "resname 3JW and (name CAR or name CAS or name NAT or name C5 or name C6 or name C4 or name N3 or name C2 or name N1)",
    "NAT": "resname 3JW and name NAT",
    "N3": "resname 3JW and name N3",
    "N1": "resname 3JW and name N1"
}

# Analyze each selection
for label, sel_string in selections.items():

    ligand = u.select_atoms(sel_string)
    print(f"\n{label}: {ligand.names}")

    if ligand.n_atoms == 0:
        print("Selection empty - skipped")
        continue

    counts = defaultdict(int)

    for ts in u.trajectory:
        contacts = set()

        for res in protein.residues:
            if distance_array(res.atoms.positions, ligand.positions).min() <= cutoff:
                contacts.add((res.resid, res.resname))

        for key in contacts:
            counts[key] += 1

    # Results
    df = pd.DataFrame(
        [(resid, resname, n, n / total_frames * 100)
         for (resid, resname), n in counts.items()],
        columns=["Resid", "Residue", "Frames", "Occupancy_percent"]
    ).sort_values("Occupancy_percent", ascending=False)

    # Save
    outfile = f"3JW_{label}_occupancy_3.5A.csv"
    df.to_csv(outfile, index=False)

    print(df.head(20))
    print(f"Saved: {outfile}")

/home/thsti/miniconda3/envs/mda311/lib/python3.11/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"



Whole: ['CAA' 'CAB' 'SAC' 'OAY' 'OAZ' 'NAD' 'CAJ' 'CAE' 'CAF' 'CAG' 'CAH' 'NAI'
 'NAK' 'NAO' 'CAN' 'CAL' 'CAM' 'C4' 'N3' 'C2' 'N1' 'C6' 'NAT' 'CAS' 'CAR'
 'C5']
    Resid Residue  Frames  Occupancy_percent
7      89     TYR     500              100.0
14     88     GLU     500              100.0
12    141     LEU     500              100.0
11     90     LEU     500              100.0
4      21     VAL     499               99.8
0      38     ALA     495               99.0
8      13     LEU     494               98.8
1     139     ASN     492               98.4
2     152     ASP     492               98.4
6     151     GLY     466               93.2
10     69     VAL     465               93.0
3      14     GLY     447               89.4
9      15     LYS     444               88.8
13    138     ARG     436               87.2
15     87     MET     403               80.6
16     16     GLY     387               77.4
17     20     SER     251               50.2
18     93     GLY     246   

In [5]:
import pandas as pd
import re

# ============================================================
# INPUT FILES
# ============================================================

occ_files = {
    "Ligand": "3JW_Whole_occupancy_3.5A.csv",
    "Scaffold": "3JW_Selected_occupancy_3.5A.csv",
    "NAT": "3JW_NAT_occupancy_3.5A.csv",
    "N3": "3JW_N3_occupancy_3.5A.csv",
    "N1": "3JW_N1_occupancy_3.5A.csv"
}

energy_file = "6wto_Delta_Energy_Decomposition.csv"
energy_file = "6wtn_Delta_Energy_Decomposition.csv"
energy_file = "6bbv_Delta_Energy_Decomposition.csv"


# ============================================================
# LOAD ENERGY DATA
# ============================================================

energy_raw = pd.read_csv(energy_file)

energy_data = []

for _, row in energy_raw.iterrows():

    m = re.match(
        r'([A-Za-z]+)\s*(-?\d+)',
        str(row["Residue_ID"]).strip()
    )

    if m:
        resname = m.group(1).upper()
        resid = m.group(2)

        energy_data.append([
            f"{resid}_{resname}",
            row["Total_Avg"],
            row["Total_StdDev"]
        ])

energy = pd.DataFrame(
    energy_data,
    columns=["key", "Energy", "StdDev"]
)

energy["Energy"] = pd.to_numeric(
    energy["Energy"], errors="coerce"
)

energy["StdDev"] = pd.to_numeric(
    energy["StdDev"], errors="coerce"
)

energy = energy.dropna(subset=["Energy"])

energy = energy.groupby(
    "key", as_index=False
).mean()


# ============================================================
# HOTSPOT CALCULATION
# ============================================================

def calculate_hotspots(occ_file, condition):

    occ = pd.read_csv(occ_file)

    occ = occ[
        ["Resid", "Residue", "Occupancy_percent"]
    ].copy()

    occ["Resid"] = occ["Resid"].astype(str).str.strip()
    occ["Residue"] = (
        occ["Residue"].astype(str).str.upper().str.strip()
    )

    occ["key"] = (
        occ["Resid"] + "_" + occ["Residue"]
    )

    occ["Occupancy_percent"] = pd.to_numeric(
        occ["Occupancy_percent"],
        errors="coerce"
    )

    occ = occ.dropna(
        subset=["Occupancy_percent"]
    )

    # Merge occupancy + energy
    merged = pd.merge(
        occ,
        energy,
        on="key",
        how="inner"
    )

    # Hotspot score
    merged["OccFraction"] = (
        merged["Occupancy_percent"] / 100
    )

    merged["HotspotScore"] = (
        merged["OccFraction"] *
        merged["Energy"].abs()
    )

    # Sort
    merged = merged.sort_values(
        "HotspotScore",
        ascending=False
    )

    # Condition
    merged.insert(
        0,
        "Condition",
        condition
    )

    # Save
    outfile = (
        f"3JW_hotspot_residues_{condition}.csv"
    )

    merged.to_csv(
        outfile,
        index=False
    )

    print(f"\n========== {condition} ==========")
    print(f"Matched residues: {len(merged)}")
    print(f"Saved: {outfile}")

    print(
        merged[
            [
                "Resid",
                "Residue",
                "Occupancy_percent",
                "Energy",
                "StdDev",
                "HotspotScore"
            ]
        ].head(20).to_string(index=False)
    )

    return merged


# ============================================================
# RUN ALL FIVE CONDITIONS
# ============================================================

results = []

for condition, occ_file in occ_files.items():

    results.append(
        calculate_hotspots(
            occ_file,
            condition
        )
    )


# ============================================================
# COMBINE RESULTS
# ============================================================

all_results = pd.concat(
    results,
    ignore_index=True
)

all_results.to_csv(
    "3JW_hotspot_residues_ALL.csv",
    index=False
)

print("\n==========================================")
print("ALL CONDITIONS COMPLETED")
print("==========================================")

print("\nCombined file:")
print("3JW_hotspot_residues_ALL.csv")

print("\nResidues matched:")
print(
    all_results.groupby("Condition").size()
)


========== Ligand ==========
Matched residues: 24
Saved: 3JW_hotspot_residues_Ligand.csv
Resid Residue  Occupancy_percent    Energy   StdDev  HotspotScore
  141     LEU              100.0 -2.785142 0.307312      2.785142
   21     VAL               99.8 -2.419540 0.477719      2.414701
   90     LEU              100.0 -1.950299 0.346982      1.950299
   89     TYR              100.0 -1.899244 0.300131      1.899244
   88     GLU              100.0 -1.702271 0.332951      1.702271
   13     LEU               98.8 -1.677255 0.390929      1.657128
  152     ASP               98.4  0.945678 0.769709      0.930547
   38     ALA               99.0 -0.936583 0.200292      0.927218
  151     GLY               93.2 -0.924587 0.254438      0.861715
   14     GLY               89.4 -0.906180 0.455066      0.810125
   15     LYS               88.8 -0.826747 0.694688      0.734152
  138     ARG               87.2 -0.768990 0.494117      0.670559
  139     ASN               98.4 -0.636057 0.380558 

In [6]:
import pandas as pd

# ============================================================
# INPUT FILES
# ============================================================

files = {
    "Ligand": "3JW_hotspot_residues_Whole.csv",
    "scaffold": "3JW_hotspot_residues_Selected.csv",
    "NAT": "3JW_hotspot_residues_NAT.csv",
    "N3": "3JW_hotspot_residues_N3.csv",
    "N1": "3JW_hotspot_residues_N1.csv"
}

energy_col = "Energy"
occ_col = "Occupancy_percent"

# ============================================================
# PROCESS ALL CONDITIONS
# ============================================================

summary = []

for condition, filename in files.items():

    df = pd.read_csv(filename)

    df[energy_col] = pd.to_numeric(
        df[energy_col], errors="coerce"
    )

    df[occ_col] = pd.to_numeric(
        df[occ_col], errors="coerce"
    )

    # Occupancy > 30%
    filtered = df[df[occ_col] > 30].copy()

    # Sum energy
    energy_sum = filtered[energy_col].sum()

    summary.append([
        condition,
        filename,
        len(filtered),
        energy_sum
    ])

    print(f"\n{condition}")
    print(f"File          : {filename}")
    print(f"Residues kept : {len(filtered)}")
    print(f"Total Energy  : {energy_sum:.6f}")


# ============================================================
# SUMMARY TABLE
# ============================================================

summary = pd.DataFrame(
    summary,
    columns=[
        "Condition",
        "File",
        "Residues_kept",
        "Total_Energy"
    ]
)

# ============================================================
# SAVE OUTPUT
# ============================================================

output_file = "filtered_energy_sums_ALL_conditions.csv"

summary.to_csv(
    output_file,
    index=False
)

# ============================================================
# PRINT FINAL SUMMARY
# ============================================================

print("\n================================================")
print("ENERGY SUM FOR OCCUPANCY > 30%")
print("================================================")

print(
    summary.to_string(index=False)
)

print("\n================================================")
print(f"Result saved as: {output_file}")
print("================================================")


Ligand
File          : 3JW_hotspot_residues_Whole.csv
Residues kept : 19
Total Energy  : -20.196613

scaffold
File          : 3JW_hotspot_residues_Selected.csv
Residues kept : 10
Total Energy  : -15.127860

NAT
File          : 3JW_hotspot_residues_NAT.csv
Residues kept : 6
Total Energy  : -9.885754

N3
File          : 3JW_hotspot_residues_N3.csv
Residues kept : 3
Total Energy  : -4.996751

N1
File          : 3JW_hotspot_residues_N1.csv
Residues kept : 5
Total Energy  : -9.248523

ENERGY SUM FOR OCCUPANCY > 30%
Condition                              File  Residues_kept  Total_Energy
   Ligand    3JW_hotspot_residues_Whole.csv             19    -20.196613
 scaffold 3JW_hotspot_residues_Selected.csv             10    -15.127860
      NAT      3JW_hotspot_residues_NAT.csv              6     -9.885754
       N3       3JW_hotspot_residues_N3.csv              3     -4.996751
       N1       3JW_hotspot_residues_N1.csv              5     -9.248523

Result saved as: filtered_energy_sums_ALL_co

In [19]:
ls

6bbv_D7D_Ligand_occupancy_3.5A.csv*
6bbv_D7D_N1_occupancy_3.5A.csv*
6bbv_D7D_N4_occupancy_3.5A.csv*
6bbv_D7D_N_occupancy_3.5A.csv*
6bbv_D7D_Scaffold_occupancy_3.5A.csv*
6bbv_Delta_Energy_Decomposition.csv*
6bbv_hotspot_Ligand.csv*
6bbv_hotspot_N1.csv*
6bbv_hotspot_N4.csv*
6bbv_hotspot_N.csv*
6bbv_hotspot_Scaffold.csv*
6bbv_no_water.prmtop*
6bbv_pca_minima1.dcd*
6WTN_Delta_Energy_Decomposition.csv*
6wtn_hotspot_Ligand.csv*
6wtn_hotspot_NAM.csv*
6wtn_hotspot_NAN.csv*
6wtn_hotspot_NAP.csv*
6wtn_hotspot_Scaffold.csv*
6wtn_no_water.prmtop*
6wtn_pca_minima1.dcd*
6wtn_RXT_Ligand_occupancy_3.5A.csv*
6wtn_RXT_NAM_occupancy_3.5A.csv*
6wtn_RXT_NAN_occupancy_3.5A.csv*
6wtn_RXT_NAP_occupancy_3.5A.csv*
6wtn_RXT_Scaffold_occupancy_3.5A.csv*
6wto_3JW_Ligand_occupancy_3.5A.csv*
6wto_3JW_N1_occupancy_3.5A.csv*
6wto_3JW_N3_occupancy_3.5A.csv*
6wto_3JW_NAT_occupancy_3.5A.csv*
6wto_3JW_Scaffold_occupancy_3.5A.csv*
6wto_Delta_Energy_Decomposition.csv*
6wto_hotspot_Ligand.csv*
6wto_hotspot_N1.csv*
6wto_hotsp

In [18]:
import MDAnalysis as mda
import pandas as pd
from collections import defaultdict
from MDAnalysis.lib.distances import distance_array

cutoff = 3.5

systems = {
"6wto":("6wto_no_water.prmtop","6wto_pca_minima1.dcd","3JW",
"6wto_Delta_Energy_Decomposition.csv",
{"Ligand":"not name H*","Scaffold":"name CAR CAS NAT C5 C6 C4 N3 C2 N1",
"NAT":"name NAT","N3":"name N3","N1":"name N1"}),

"6wtn":("6wtn_no_water.prmtop","6wtn_pca_minima1.dcd","RXT",
"6WTN_Delta_Energy_Decomposition.csv",
{"Ligand":"not name H*","Scaffold":"name CAC CAD CAE CAR CAS CAT NAM NAN NAP",
"NAP":"name NAP","NAM":"name NAM","NAN":"name NAN"}),

"6bbv":("6bbv_no_water.prmtop","6bbv_pca_minima1.dcd","D7D",
"6bbv_Delta_Energy_Decomposition.csv",
{"Ligand":"not name H*","Scaffold":"name N C C1 N1 C2 C3 N4 C12 C13",
"N":"name N","N1":"name N1","N4":"name N4"})
}

residue_all, total_all = [], []

for system,(top,traj,resname,energy_file,selections) in systems.items():

    u = mda.Universe(top,traj)
    protein = u.select_atoms("protein")
    nframes = len(u.trajectory)

    e = pd.read_csv(energy_file)
    x = e["Residue_ID"].astype(str).str.extract(
        r'([A-Za-z]+)\s*(-?\d+)',expand=True)
    e["key"] = x[1]+"_"+x[0].str.upper()
    e = e[["key","Total_Avg","Total_StdDev"]].rename(
        columns={"Total_Avg":"Energy","Total_StdDev":"StdDev"})

    for condition,selection in selections.items():

        ligand = u.select_atoms(f"resname {resname} and ({selection})")
        counts = defaultdict(int)

        for ts in u.trajectory:
            contacts = set()
            for res in protein.residues:
                if distance_array(res.atoms.positions,
                                  ligand.positions).min() <= cutoff:
                    contacts.add((res.resid,res.resname))
            for key in contacts:
                counts[key] += 1

        occ = pd.DataFrame(
            [(r,rn,n,n/nframes*100) for (r,rn),n in counts.items()],
            columns=["Resid","Residue","Frames","Occupancy_percent"])

        occ.to_csv(
            f"{system}_{resname}_{condition}_occupancy_3.5A.csv",
            index=False)

        occ["key"] = occ["Resid"].astype(str)+"_"+occ["Residue"].str.upper()
        df = occ.merge(e,on="key",how="inner")

        df["HotspotScore"] = (
            df["Occupancy_percent"]/100 * df["Energy"].abs())

        df = df.sort_values("HotspotScore",ascending=False)
        df.to_csv(f"{system}_hotspot_{condition}.csv",index=False)

        f = df[df["Occupancy_percent"]>30].copy()
        f.insert(0,"System",system)
        f.insert(1,"Condition",condition)

        residue_all.append(f[
            ["System","Condition","Resid","Residue",
             "Occupancy_percent","Energy","StdDev","HotspotScore"]])

        total_all.append([
            system,condition,len(f),f["Energy"].sum()])

    del u

# Residue-level summary
residue_summary = pd.concat(residue_all,ignore_index=True)
residue_summary = residue_summary.sort_values(
    ["System","Condition","HotspotScore"],
    ascending=[True,True,False])

residue_summary.to_csv(
    "FINAL_residue_summary_occupancy_gt30.csv",index=False)

# Total-energy summary
total_summary = pd.DataFrame(
    total_all,
    columns=["System","Condition","Residues_kept","Total_Energy"])

total_summary.to_csv(
    "FINAL_energy_summary_occupancy_gt30.csv",index=False)

print("\nResidue-level summary:")
print(residue_summary.to_string(index=False))

print("\nTotal-energy summary:")
print(total_summary.to_string(index=False))

/home/thsti/miniconda3/envs/mda311/lib/python3.11/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"
/home/thsti/miniconda3/envs/mda311/lib/python3.11/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"
/home/thsti/miniconda3/envs/


Residue-level summary:
System Condition  Resid Residue  Occupancy_percent    Energy   StdDev  HotspotScore
  6bbv    Ligand     90     LEU              100.0 -2.104370 0.422177      2.104370
  6bbv    Ligand    141     LEU              100.0 -2.056852 0.405245      2.056852
  6bbv    Ligand     89     TYR              100.0 -1.992603 0.288653      1.992603
  6bbv    Ligand     21     VAL               97.8 -1.920356 0.412551      1.878108
  6bbv    Ligand     13     LEU               98.0 -1.797378 0.507700      1.761431
  6bbv    Ligand     88     GLU              100.0 -1.568248 0.288889      1.568248
  6bbv    Ligand     14     GLY               96.0 -1.002143 0.404447      0.962057
  6bbv    Ligand     38     ALA               94.2 -0.817234 0.134606      0.769834
  6bbv    Ligand     93     GLY               86.2 -0.722255 0.241136      0.622583
  6bbv    Ligand     94     SER               64.6 -0.619768 0.171547      0.400370
  6bbv    Ligand     87     MET               66.2 -